# Track A: the quantum algorithm, step by step

Run the classical notebook first. **S only, 14 atoms, local emu-sv, raw shots,
no repair.** The target QUBO, decoder, exhaustive checks, atom placement, pulse,
calibration, and benchmark loop are defined in the cells below.

Only validated input loading, persistence, routing-based calibration metrics,
device-independent reporting, and the resource-limited emulator subprocess
remain shared. The local pulse is passed directly into that subprocess;
the notebook does not launch the hidden CLI pipeline.

## Before the code: bits, atoms and measurements

Start with the classical notebook's problem: decide which vehicle serves each
customer. A **bit** is a value of 0 or 1. For each customer we use two assignment
bits: `[1, 0]` means vehicle 0, and `[0, 1]` means vehicle 1. `[0, 0]` and
`[1, 1]` break the "exactly one vehicle" rule.

A **neutral atom** has no net electric charge. A real device holds atoms
at chosen positions using laser traps; the emulator models them at fixed
positions in a plane. Each bit is represented by one atom.
Lasers control the atom's two modeled states: ground state (`0`) and a highly
excited **Rydberg** state (`1`). A **qubit** is this quantum two-state system.
During evolution it is not generally restricted to one classical bit value;
a final **measurement** produces an ordinary 0/1 string.

Nearby excited atoms repel energetically. We choose their positions and laser
controls to make useful answer strings have lower energy. The laser time
pattern is a **pulse**. A **Hamiltonian** is the mathematical description of
the system's energy and its evolution; its lowest-energy state is called a
**ground state**. Trying to favor a low-energy answer does not guarantee that
a finite pulse finds the correct answer.

One measurement is a **shot**. Repeating a run 256 times gives a table of
bitstrings and counts, not one guaranteed solution. We use a local computer
simulation (**emulation**), not a physical quantum processor. The extra bits
introduced for capacity are explained below; not every atom represents a
customer.

In [1]:
from pathlib import Path
import json
import sys
from time import perf_counter
import numpy as np
import pandas as pd
from IPython.display import display

root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "track_a" / "inputs" / "seeds_s.json").exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from track_a.common.instances import OUTPUT_PATH, Instance, ensure_instances, feasible, validate
from track_a.common.analysis import fingerprint, save
from track_a.common.models import ENCODING

instances = ensure_instances()
print("Processing only the first demand vector:", instances[0].demands)
output = OUTPUT_PATH
shots = 256
seed = 0

from dataclasses import asdict, dataclass
from math import ceil, log2
from numpy.typing import NDArray
from pulser import AnalogDevice, Pulse, Register, Sequence
from pulser.waveforms import InterpolatedWaveform, RampWaveform
from track_a.common.classical import Routing
from track_a.common.analysis import metrics, plot_evidence
from track_a.common.quantum import emulate_sequence

timeout_seconds = 180
memory_mb = 4096

Processing only the first demand vector: (2, 3, 4, 2)


## 1. State the objective and bitstring representation

Compactness sums customer distances within each vehicle, not route
length. Assignment bits are customer-major (`x_0_0`, `x_0_1`, ...); little-endian
slack bits follow. The QUBO stores a symmetric matrix plus its constant.

**Why a matrix?** A QUBO (quadratic
unconstrained binary optimization) assigns a score to a string of bits.
It contains terms for individual bits and for pairs of bits, but not triples.
The matrix is just a convenient way to store those coefficients. A constant
adds the same amount to every score, so it cannot change which answer wins.

In [2]:
def objective(instance: Instance, labels: tuple[int, ...] | list[int]) -> float:
    distances = instance.distances
    return float(sum(distances[i, j] for i in range(instance.n)
                     for j in range(i) if labels[i] == labels[j]))


def canonical(labels: tuple[int, ...] | list[int]) -> tuple[int, ...]:
    mapping: dict[int, int] = {}
    return tuple(mapping.setdefault(k, len(mapping)) for k in labels)


@dataclass
class Qubo:
    matrix: NDArray[np.float64]
    constant: float
    penalty: float
    normalization: float
    slack_bits: int
    variable_order: list[str]

    def energy(self, bits: NDArray | list[int]) -> float:
        z = np.asarray(bits)
        return float(z @ self.matrix @ z + self.constant)

## 2. Expand assignment and capacity penalties into Q

With $D=\max(1,\max c_{ij})$ and
$s_k=\sum_{r=0}^{2}2^r y_{kr}$, minimize
$$E=F/D+A\sum_i(\sum_kx_{ik}-1)^2+
B\sum_k(\sum_i d_ix_{ik}+s_k-Q)^2.$$
If U is the normalized feasible-witness cost, **A=B=U+1** suffices:
every nonzero integer residual costs more than U and F is nonnegative.
This proves the exact target ground states, not the physical embedding.
The `square` function below explicitly uses $z_i^2=z_i$.
Pair coefficients are **twice** off-diagonal Q entries.

**A penalty turns a rule into a cost.** If a
customer has zero or two selected vehicle bits, `(sum - 1)^2` is positive;
if it has exactly one, the penalty is zero. Squaring makes violations cost
something whichever direction they go.

**Slack means unused carrying capacity.** A vehicle carrying 5 units has
slack `7 - 5 = 2`. Three binary slack bits have weights 1, 2, 4, so the
little-endian string `[0, 1, 0]` represents 2. Adding slack changes the
"load at most 7" rule into "load + slack equals 7." An overloaded vehicle
would need negative slack, which these bits cannot represent.

**Why the large weights A and B?** They make breaking a rule more expensive
than the entire score of a known legal answer. D rescales distances into
smaller numbers, and U is that legal answer's rescaled score. These are
mathematical penalty settings, not laser settings.

In [3]:
def build_qubo(instance: Instance, penalty: float | None = None) -> Qubo:
    n, k, q = instance.n, instance.vehicles, instance.capacity
    b = ceil(log2(q + 1))
    size = n * k + k * b
    normalization = max(float(instance.distances.max()), 1.0)
    upper = objective(instance, instance.certificate) / normalization
    weight = upper + 1 if penalty is None else penalty
    if not np.isfinite(weight) or weight <= upper:
        raise ValueError("Penalty must exceed feasible-certificate normalized objective")
    matrix = np.zeros((size, size))
    for i in range(n):
        for j in range(i):
            for vehicle in range(k):
                a, c = i * k + vehicle, j * k + vehicle
                matrix[a, c] += instance.distances[i, j] / normalization / 2
                matrix[c, a] = matrix[a, c]
    constant = 0.0

    def square(coefficients: NDArray[np.float64], rhs: int) -> None:
        nonlocal constant
        matrix[:] += weight * np.outer(coefficients, coefficients)
        matrix[np.diag_indices(size)] -= 2 * weight * rhs * coefficients
        constant += weight * rhs**2

    for i in range(n):
        coefficients = np.zeros(size)
        coefficients[i * k:(i + 1) * k] = 1
        square(coefficients, 1)
    for vehicle in range(k):
        coefficients = np.zeros(size)
        coefficients[vehicle:n * k:k] = instance.demands
        coefficients[n * k + vehicle * b:n * k + (vehicle + 1) * b] = 2.0 ** np.arange(b)
        square(coefficients, q)
    names = [f"x_{i}_{v}" for i in range(n) for v in range(k)]
    names += [f"y_{v}_{bit}" for v in range(k) for bit in range(b)]
    return Qubo(matrix, constant, weight, normalization, b, names)

In [4]:
example = instances[0]
qubo = build_qubo(example)
print("Assignment atoms:", example.n * example.vehicles)
print("Slack atoms:", example.vehicles * qubo.slack_bits)
print("Total atoms:", len(qubo.variable_order), "penalty:", qubo.penalty)
display(pd.DataFrame(qubo.matrix, index=qubo.variable_order, columns=qubo.variable_order))

Assignment atoms: 8
Slack atoms: 6
Total atoms: 14 penalty: 2.508126572372197


,x_0_0,x_0_1,x_1_0,x_1_1,x_2_0,x_2_1,x_3_0,x_3_1,y_0_0,y_0_1,y_0_2,y_1_0,y_1_1,y_1_2
x_0_0,-62.703164,2.508127,15.373102,0.000000,20.391361,0.000000,10.099603,0.000000,5.016253,10.032506,20.065013,0.000000,0.000000,0.000000
x_0_1,2.508127,-62.703164,0.000000,15.373102,0.000000,20.391361,0.000000,10.099603,0.000000,0.000000,0.000000,5.016253,10.032506,20.065013
x_1_0,15.373102,0.000000,-85.276303,2.508127,30.597519,0.000000,15.411383,0.000000,7.524380,15.048759,30.097519,0.000000,0.000000,0.000000
x_1_1,0.000000,15.373102,2.508127,-85.276303,0.000000,30.597519,0.000000,15.411383,0.000000,0.000000,0.000000,7.524380,15.048759,30.097519
x_2_0,20.391361,0.000000,30.597519,0.000000,-102.833189,2.508127,20.329126,0.000000,10.032506,20.065013,40.130025,0.000000,0.000000,0.000000
x_2_1,0.000000,20.391361,0.000000,30.597519,2.508127,-102.833189,0.000000,20.329126,0.000000,0.000000,0.000000,10.032506,20.065013,40.130025
x_3_0,10.099603,0.000000,15.411383,0.000000,20.329126,0.000000,-62.703164,2.508127,5.016253,10.032506,20.065013,0.000000,0.000000,0.000000
x_3_1,0.000000,10.099603,0.000000,15.411383,0.000000,20.329126,2.508127,-62.703164,0.000000,0.000000,0.000000,5.016253,10.032506,20.065013
y_0_0,5.016253,0.000000,7.524380,0.000000,10.032506,0.000000,5.016253,0.000000,-32.605645,5.016253,10.032506,0.000000,0.000000,0.000000
y_0_1,10.032506,0.000000,15.048759,0.000000,20.065013,0.000000,10.032506,0.000000,5.016253,-60.195038,20.065013,0.000000,0.000000,0.000000


## 3. Decode the original constraints, and inspect slack separately

A usable sampled clustering must assign each customer once and fit
within capacity. It need not have correct slack bits; those are checked
separately when validating the target ground states. `encode` is only used for
classical evidence, never as a quantum warm start or repair.

**Two kinds of correctness:** a measured string
can describe a legal customer grouping even if its unused-capacity bits are
wrong. We can still evaluate that grouping. However, the mathematical claim
about the target's very best bitstrings requires both assignment and slack
rules to hold. Keeping these checks separate avoids mixing up model validation
with measurement quality.

In [5]:
def decode(instance: Instance, bitstring: str) -> tuple[int, ...] | None:
    b = ceil(log2(instance.capacity + 1))
    if len(bitstring) != instance.n * instance.vehicles + instance.vehicles * b:
        raise ValueError("Bitstring length does not match encoding")
    if set(bitstring) - {"0", "1"}:
        raise ValueError("Nonbinary candidate")
    x = np.array([int(c) for c in bitstring[:instance.n * instance.vehicles]])
    x = x.reshape(instance.n, instance.vehicles)
    if not np.all(x.sum(axis=1) == 1):
        return None
    labels = tuple(int(v) for v in x.argmax(axis=1))
    return canonical(labels) if feasible(instance, labels) else None


def encode(instance: Instance, labels: tuple[int, ...]) -> str:
    if not feasible(instance, labels):
        raise ValueError("Cannot encode infeasible assignment")
    b = ceil(log2(instance.capacity + 1))
    bits = [int(v == labels[i]) for i in range(instance.n) for v in range(instance.vehicles)]
    for v in range(instance.vehicles):
        slack = instance.capacity - sum(d for i, d in enumerate(instance.demands) if labels[i] == v)
        bits.extend((slack >> bit) & 1 for bit in range(b))
    return "".join(map(str, bits))


def residuals(instance: Instance, bitstring: str) -> tuple[list[int], list[int]]:
    decode(instance, bitstring)
    k, n = instance.vehicles, instance.n
    bits = np.array(list(map(int, bitstring)))
    x = bits[:n * k].reshape(n, k)
    b = (len(bits) - n * k) // k
    slack = bits[n * k:].reshape(k, b) @ (2 ** np.arange(b))
    return (x.sum(axis=1) - 1).tolist(), (np.array(instance.demands) @ x + slack - instance.capacity).tolist()


def bit_layout(instance: Instance, target: Qubo) -> list[dict]:
    assignment_size = instance.n * instance.vehicles
    rows = []
    for index, name in enumerate(target.variable_order):
        is_slack = index >= assignment_size
        offset = index - assignment_size
        rows.append({
            "position_from_left": index + 1, "python_index": index,
            "variable": name, "kind": "slack" if is_slack else "assignment",
            "vehicle": offset // target.slack_bits if is_slack else index % instance.vehicles,
            "customer": None if is_slack else index // instance.vehicles,
            "slack_weight": 2 ** (offset % target.slack_bits) if is_slack else None,
        })
    return rows


def bitstring_breakdown(instance: Instance, bitstring: str) -> dict:
    decode(instance, bitstring)
    assignment_size = instance.n * instance.vehicles
    width = ceil(log2(instance.capacity + 1))
    result = {"assignment_bits": bitstring[:assignment_size],
              "slack_bits": bitstring[assignment_size:]}
    groups = [bitstring[:assignment_size]]
    for vehicle in range(instance.vehicles):
        start = assignment_size + vehicle * width
        bits = bitstring[start:start + width]
        result[f"slack_bits_vehicle_{vehicle}"] = bits
        result[f"slack_value_vehicle_{vehicle}"] = sum(int(bit) * 2**b for b, bit in enumerate(bits))
        groups.append(bits)
    result["grouped_bitstring"] = " | ".join(groups)
    return result

## 4. Check all 16,384 target bitstrings against the exact reference

Every ground state must satisfy both assignment and slack residuals
and match the constrained compactness optimum. Vehicle-label symmetries are
counted as one decoded partition.

**Why exhaustive checking is possible here:** 14 bits
give `2**14 = 16,384` possible strings. We score all of them, find the lowest
score, and check the associated groupings against the exact classical answer.
This validates the intended score function before asking whether atom physics
can represent it well.

In [6]:
def validate_small_qubo(instance: Instance, reference: float) -> dict:
    qubo = build_qubo(instance)
    size = len(qubo.variable_order)
    if size > 16:
        raise ValueError("Exhaustive QUBO validation is restricted to small instances")
    bits = ((np.arange(2**size)[:, None] >> np.arange(size)) & 1)
    energies = np.einsum("bi,ij,bj->b", bits, qubo.matrix, bits) + qubo.constant
    ground = np.flatnonzero(np.isclose(energies, energies.min(), atol=1e-8, rtol=0))
    partitions = set()
    for index in ground:
        bitstring = "".join(map(str, bits[index]))
        labels = decode(instance, bitstring)
        assignment, capacity = residuals(instance, bitstring)
        if labels is None or any(assignment + capacity):
            raise AssertionError("Infeasible target QUBO ground state")
        if not np.isclose(objective(instance, labels), reference):
            raise AssertionError("QUBO and constrained optima disagree")
        partitions.add(labels)
    return {"checked_bitstrings": 2**size, "ground_states": len(ground),
            "ground_state_bitstrings": ["".join(map(str, bits[index])) for index in ground],
            "distinct_ground_partitions": len(partitions), "passed": True,
            "minimum_energy": float(energies.min())}

In [7]:
checks = []
for instance in instances:
    reference = json.loads((output / "references" / f"{instance.id}.json").read_text())
    if reference["instance_hash"] != fingerprint(asdict(instance)):
        raise ValueError("Classical references do not match the current input")
    checks.append({"instance": instance.id,
                   **validate_small_qubo(instance, reference["clustering"]["cost"])})
display(pd.DataFrame(checks))
ground_rows = []
for instance, check in zip(instances, checks, strict=True):
    target = build_qubo(instance)
    print("Bit positions, left to right:", target.variable_order)
    display(pd.DataFrame(bit_layout(instance, target)))
    for bitstring in check["ground_state_bitstrings"]:
        ground_rows.append({
            "instance": instance.id, "bitstring": bitstring,
            **bitstring_breakdown(instance, bitstring),
            "target_energy": target.energy(list(map(int, bitstring))),
            "customer_groups": decode(instance, bitstring),
        })
    save(output / "groundstates" / f"{instance.id}-exact.json", {
        "instance_id": instance.id, "instance_hash": fingerprint(asdict(instance)),
        "variable_order": target.variable_order, "bit_layout": bit_layout(instance, target),
        "ground_state_details": [row for row in ground_rows if row["instance"] == instance.id],
        **check,
    })
print("Exact target-QUBO ground states (not necessarily observed in quantum shots):")
display(pd.DataFrame(ground_rows))

,instance,checked_bitstrings,ground_states,ground_state_bitstrings,distinct_ground_partitions,passed,minimum_energy
0,s-0,16384,2,"[10010110110000, 01101001000110]",1,True,1.134194


Bit positions, left to right: ['x_0_0', 'x_0_1', 'x_1_0', 'x_1_1', 'x_2_0', 'x_2_1', 'x_3_0', 'x_3_1', 'y_0_0', 'y_0_1', 'y_0_2', 'y_1_0', 'y_1_1', 'y_1_2']


,position_from_left,python_index,variable,kind,vehicle,customer,slack_weight
0,1,0,x_0_0,assignment,0,0.0,NaN
1,2,1,x_0_1,assignment,1,0.0,NaN
2,3,2,x_1_0,assignment,0,1.0,NaN
3,4,3,x_1_1,assignment,1,1.0,NaN
4,5,4,x_2_0,assignment,0,2.0,NaN
5,6,5,x_2_1,assignment,1,2.0,NaN
6,7,6,x_3_0,assignment,0,3.0,NaN
7,8,7,x_3_1,assignment,1,3.0,NaN
8,9,8,y_0_0,slack,0,NaN,1.0
9,10,9,y_0_1,slack,0,NaN,2.0


Exact target-QUBO ground states (not necessarily observed in quantum shots):


,instance,bitstring,assignment_bits,slack_bits,slack_bits_vehicle_0,slack_value_vehicle_0,slack_bits_vehicle_1,slack_value_vehicle_1,grouped_bitstring,target_energy,customer_groups
0,s-0,10010110110000,10010110,110000,110,3,000,0,10010110 | 110 | 000,1.134194,"(0, 1, 1, 0)"
1,s-0,01101001000110,01101001,000110,000,0,110,3,01101001 | 000 | 110,1.134194,"(0, 1, 1, 0)"


## 5. Check the installed hardware envelope

Use Pulser's full-precision limits rather than rounded PDF numbers.
The interaction coefficient already includes conversion to angular-frequency
units. Never relax device constraints to make a layout succeed.

**Units and limits:** atom coordinates are in
micrometers (um). Pulse durations are in nanoseconds (ns); 1000 ns is one
microsecond (us). Laser controls and interaction strengths use angular
frequency in radians per microsecond (rad/us). Do not mix these with ordinary
frequency in MHz without the factor of `2*pi`. Hardware bounds limit atom
spacing, register size, laser strength, detuning and duration.

In [8]:
def device_metadata() -> dict:
    channel = AnalogDevice.channels["rydberg_global"]
    actual = {
        "max_atoms": AnalogDevice.max_atom_num,
        "min_separation_um": AnalogDevice.min_atom_distance,
        "max_radius_um": AnalogDevice.max_radial_distance,
        "max_amplitude": channel.max_amp,
        "max_abs_detuning": channel.max_abs_detuning,
        "max_duration_ns": AnalogDevice.max_sequence_duration,
        "interaction_coeff": AnalogDevice.interaction_coeff,
    }
    guide = [80, 5, 38, 12.57, 125.66, 6000, 8.657e5]
    if not all(np.isclose(a, b, rtol=5e-4) for a, b in zip(actual.values(), guide, strict=True)):
        raise RuntimeError(f"Installed AnalogDevice differs from challenge guide: {actual}")
    return {**actual, "guide_discrepancy_beyond_rounding": False,
            "abstract_device": json.loads(AnalogDevice.to_abstract_repr())}

## 6. Fit a legal variable-to-atom mapping

At zero drive,
$E_{\rm native}=-\delta_f\sum_a z_a+\sum_{a<b}(C_6/\hbar)r_{ab}^{-6}z_az_b$.
Greedy seeded swaps align a legal triangular register with target pair
coefficients. Fit positive scale s, use the mean scaled diagonal for the
shared detuning, and save **all** interactions, including target zeros.
Unequal linear terms and 2D geometry make this an approximation:
$e_J=\|J_{\rm real}-sJ_{\rm target}\|_F/\|sJ_{\rm target}\|_F$.
The corresponding linear error is reported separately.

**Embedding means choosing where each variable's atom
goes.** The customer map in the input is not the atom map: customer distances
define the delivery problem; atom distances define the simulated interactions.
Pairs of excited atoms interact much more strongly when close together:
doubling their separation weakens the interaction by a factor of 64.

J is the table of desired pair interactions; "real" means what our atom layout
actually produces. The positive scale s changes the overall energy units.
The error compares the mismatch with the size of the intended interactions.
Zero would be perfect; a larger value is worse. It is not a percentage of
correct answers or a probability of success.

There is one shared final laser detuning, but the QUBO usually wants different
individual-bit coefficients. Also, arbitrary pair strengths cannot all be
realized just by moving points in a plane. These are real limitations, not
details we hide by dropping troublesome terms.

In [9]:
def embedding(qubo: Qubo, seed: int, swaps: int = 1200) -> dict:
    """Fit atom permutation on a legal triangular grid, then a positive scale."""
    start = perf_counter()
    device_metadata()
    size = len(qubo.variable_order)
    if size > AnalogDevice.max_atom_num:
        raise ValueError("QUBO exceeds AnalogDevice atom count")
    pairs = 2 * (qubo.matrix - np.diag(np.diag(qubo.matrix)))
    if np.any(pairs < 0):
        raise ValueError("Negative pair couplings are unsupported by repulsive Rydberg interactions")
    candidates = np.array([(i + j / 2, np.sqrt(3) * j / 2)
                           for i in range(-size, size + 1)
                           for j in range(-size, size + 1)])
    candidates = candidates[np.argsort(np.linalg.norm(candidates, axis=1), kind="stable")[:size]]
    coordinates = (candidates - candidates.mean(axis=0)) * 8.0
    radius = np.linalg.norm(coordinates, axis=1).max()
    if radius > 37.9:
        coordinates *= 37.9 / radius
    distances = np.linalg.norm(coordinates[:, None] - coordinates[None, :], axis=-1)
    np.fill_diagonal(distances, np.inf)
    interactions = AnalogDevice.interaction_coeff / distances**6
    rng = np.random.default_rng(seed)
    order = rng.permutation(size)
    norm = np.linalg.norm(pairs)
    if norm == 0:
        raise ValueError("Cannot fit a zero interaction target")

    def score(permutation: np.ndarray) -> float:
        realized = interactions[np.ix_(permutation, permutation)]
        return float(np.sum(realized * pairs))

    best = score(order)
    for _ in range(swaps):
        a, b = rng.choice(size, 2, replace=False)
        order[a], order[b] = order[b], order[a]
        value = score(order)
        if value > best:
            best = value
        else:
            order[a], order[b] = order[b], order[a]
    coordinates = coordinates[order]
    realized = interactions[np.ix_(order, order)]
    scale = float(np.sum(realized * pairs) / norm**2)
    linear = np.diag(qubo.matrix)
    common = float(scale * linear.mean())
    channel = AnalogDevice.channels["rydberg_global"]
    if scale <= 0 or abs(common) > channel.max_abs_detuning:
        raise ValueError("Fitted scale or final global detuning is illegal")
    register = Register(dict(zip(qubo.variable_order, coordinates, strict=True)))
    AnalogDevice.validate_register(register)
    return {
        "variable_order": qubo.variable_order, "coordinates_um": coordinates.tolist(),
        "target_pairs": pairs.tolist(), "realized_pairs": realized.tolist(),
        "target_linear": linear.tolist(), "realized_linear": [common] * size,
        "scale": scale, "constant_offset_scaled": scale * qubo.constant,
        "final_detuning": -common,
        "pair_relative_error": float(np.linalg.norm(realized - scale * pairs) / (scale * norm)),
        "linear_relative_error": float(np.linalg.norm(common - scale * linear) /
                                       np.linalg.norm(scale * linear)),
        "unsupported_coupling_signs": False, "nonquadratic_terms": False,
        "unequal_linear_terms": not np.allclose(linear, linear[0]),
        "seed": seed, "settings": {"algorithm": "triangular-grid-greedy-swaps",
                                   "swap_attempts": swaps, "initial_spacing_um": 8.0},
        "legal": True, "seconds": perf_counter() - start,
    }

In [10]:
layout = embedding(qubo, seed)
print({key: layout[key] for key in
       ("legal", "scale", "pair_relative_error", "linear_relative_error", "final_detuning")})

{'legal': True, 'scale': 0.048542822021241085, 'pair_relative_error': 1.0768493453955446, 'linear_relative_error': 0.3061499463062373, 'final_detuning': 3.5134016350715167}


## 7. Construct the actual Pulser pulse

The smooth three-point Rabi waveform starts and ends at zero.
Global detuning sweeps from negative initial detuning to the fitted final
value. Durations, frequencies, coordinates and modulation fall time are
checked against AnalogDevice.

**Two controls change over time.** Rabi
amplitude is the laser drive strength that mixes the two atomic states.
Detuning is how far the laser is shifted from the atomic transition frequency.
With the convention used here, positive detuning favors excitation (`1`);
negative detuning discourages it.

The amplitude rises and falls smoothly; detuning sweeps from negative to its
chosen final value. Zero final amplitude lets us compare the final energy
with the bitstring cost. A smooth sweep can help, but it is not a promise
of reaching the best answer.

In [11]:
@dataclass(frozen=True)
class PulseSettings:
    duration_ns: int = 1000
    amplitude: float = 8.0
    initial_detuning: float = -8.0


def sequence_for(layout: dict, settings: PulseSettings) -> Sequence:
    channel = AnalogDevice.channels["rydberg_global"]
    if (settings.duration_ns % channel.clock_period
            or not channel.min_duration <= settings.duration_ns <= AnalogDevice.max_sequence_duration
            or not 0 < settings.amplitude <= channel.max_amp
            or not -channel.max_abs_detuning <= settings.initial_detuning < 0):
        raise ValueError("Illegal pulse settings")
    register = Register(dict(zip(layout["variable_order"], layout["coordinates_um"], strict=True)))
    AnalogDevice.validate_register(register)
    sequence = Sequence(register, AnalogDevice)
    sequence.declare_channel("drive", "rydberg_global")
    amplitude = InterpolatedWaveform(settings.duration_ns, [0, settings.amplitude, 0])
    detuning = RampWaveform(settings.duration_ns, settings.initial_detuning, layout["final_detuning"])
    sequence.add(Pulse(amplitude, detuning, 0), "drive")
    sequence.measure("ground-rydberg")
    if sequence.get_duration(include_fall_time=True) > AnalogDevice.max_sequence_duration:
        raise ValueError("Pulse including modulation fall time exceeds device duration")
    return sequence

## 8. Emulate this notebook's sequence with resource protection

`emulate_sequence` is only the shared process supervisor: it takes
the **Sequence constructed above**, runs `SVBackend` with `SVConfig(dt=10,
gpu=False, interaction_cutoff=0, with_modulation=False)` and a final
`BitStrings(num_shots=shots)` observable, seeds NumPy/Torch, and enforces
wall-time/address-space limits. It returns raw counts or an explicit failure
status. It does not construct a QUBO, embed it, shape its pulse, or repair shots.
The wrapper below passes our local controls directly to that supervisor.

**Why keep a process helper?** The simulator can use
a lot of time or memory. Running it in a separate process lets us stop it
cleanly if it exceeds a limit. The visible notebook supplies the actual atom
layout and pulse. A timestep of 10 ns controls the numerical simulation;
zero interaction cutoff means even weak, unwanted pair interactions remain
in the model.

In [12]:
def emulate(layout: dict, settings: PulseSettings, shots: int, seed: int,
            timeout_seconds: float = 180, memory_mb: int = 4096) -> dict:
    result = emulate_sequence(sequence_for(layout, settings), shots, seed,
                              timeout_seconds=timeout_seconds, memory_mb=memory_mb)
    result["settings"] = {**asdict(settings), **result["settings"]}
    return result

## 9. Calibrate, freeze, then execute the saved instances

Read this loop to see precisely what is optimized: two fixed
1000/2000 ns sweeps at peak amplitude 8 are compared on S-0, using best
feasible gap, hit probability and feasibility. The selected settings freeze
before the instance loop. Calibration reuses S-0 (not held-out data).
Assignments, serialized sequences, layouts, raw counts and failure statuses
are preserved. No classical search repairs quantum outputs.

**Calibration is a small experiment before the
main comparison.** We try two pulse durations on the first saved instance,
choose one, and use it unchanged for the rest. A seed makes random choices
repeatable. Using that first benchmark instance for calibration may favor it,
so we disclose it rather than describing it as an independent test.

In [13]:
def quantum_runs(instances, output: Path, shots: int, seed: int,
                 timeout: float, memory_mb: int) -> None:
    save(output / "device.json", device_metadata())
    first = instances[0]
    ref = json.loads((output / "references" / f"{first.id}.json").read_text())
    qubo = build_qubo(first)
    validation = validate_small_qubo(first, ref["clustering"]["cost"])
    first_layout = embedding(qubo, seed)
    candidates = [PulseSettings(1000, 8.0), PulseSettings(2000, 8.0)]
    trials = []
    for pulse in candidates:
        print(f"Calibrating S: {pulse}", flush=True)
        result = emulate(first_layout, pulse, shots, seed, timeout_seconds=timeout, memory_mb=memory_mb)
        measured, _ = metrics(first, result.get("counts", {}), ref, Routing(first))
        trials.append({"pulse": asdict(pulse), "result": result, "metrics": measured})
    valid = [trial for trial in trials if trial["result"]["status"] == "completed"]
    selected = max(valid, key=lambda t: (
        -(t["metrics"]["best_model_gap"] if t["metrics"]["best_model_gap"] is not None else float("inf")),
        t["metrics"]["model_hit_probability_0.1pct"] or 0,
        t["metrics"]["feasibility_rate"] or 0,
    )) if valid else trials[0]
    pulse = PulseSettings(**selected["pulse"])
    save(output / "calibration.json", {
        "instance_id": first.id, "selected": asdict(pulse), "trials": trials,
        "frozen_before_benchmark": True, "uses_benchmark_instance": True,
        "selection": "best feasible model gap, reference hit probability, feasibility",
    })
    small_ok = validation["passed"]
    for instance in instances:
        print(f"Quantum {instance.id}", flush=True)
        record = asdict(instance)
        qubo = build_qubo(instance)
        configuration = {"pulse": asdict(pulse), "embedding_seed": seed, "sampling_seed": seed,
                         "backend": "emu-sv", "dt_ns": 10, "interaction_cutoff": 0.0,
                         "memory_mb": memory_mb, "timeout_seconds": timeout, "shots": shots,
                         "embedding_swap_attempts": 1200, "penalty": qubo.penalty,
                         "normalization": qubo.normalization, "postprocessing": False}
        base = {
            "instance": record, "instance_id": instance.id, "instance_hash": fingerprint(record),
            "method": "quantum-global", "seed": seed, "encoding": ENCODING,
            "shot_budget": shots, "atom_count": len(qubo.variable_order),
            "configuration": configuration, "configuration_id": fingerprint(configuration),
        }
        layout = first_layout if instance.id == first.id else embedding(qubo, seed)
        save(output / "embeddings" / f"{instance.id}-{seed}.json", layout)
        sequence = sequence_for(layout, pulse)
        save(output / "sequences" / f"{instance.id}-{seed}.json",
             json.loads(sequence.to_abstract_repr()))
        if instance.id == first.id:
            result = selected["result"]
        else:
            result = emulate(layout, pulse, shots, seed,
                             timeout_seconds=timeout, memory_mb=memory_mb)
        has_feasible = any(decode(instance, bits) is not None for bits in result.get("counts", {}))
        small_ok = small_ok and result["status"] == "completed" and has_feasible
        save(output / "runs" / f"{instance.id}-quantum-global-{seed}.json", {
            **base, **result, "variable_order": qubo.variable_order,
            "embedding_seconds": layout["seconds"], "emulation_seconds": result["seconds"],
            "pair_relative_error": layout["pair_relative_error"],
            "linear_relative_error": layout["linear_relative_error"], "scale": layout["scale"],
            "pulse": asdict(pulse),
        })
    save(output / "checkpoint.json", {"s_passed": small_ok,
                                     "requirements": "exact QUBO, legal device, feasible raw S shots"})

In [ ]:
instance_path = output / "instances.json"
if (not instance_path.exists()
        or fingerprint(json.loads(instance_path.read_text())) != fingerprint(
            [{**asdict(i), **i.ratios, "certificate_valid": True} for i in instances])):
    raise ValueError("Run the classical notebook with these inputs first")
quantum_runs(instances, output, shots, seed, timeout_seconds, memory_mb)
calibration = json.loads((output / "calibration.json").read_text())
print("Frozen pulse:", calibration["selected"])
plot_evidence(output, instances[0].id, seed)

## 10. Show ground-state bitstrings and the lowest-energy measured answers

A **ground-state bitstring** has the lowest possible score for the target QUBO. The exhaustive check above lists every such string, even when the quantum shots never measure it. Two strings can describe the same groups with the vehicle names swapped.

Below we show any exact target ground states actually measured, with their counts. We also show the lowest-target-energy strings measured, even if none is a true target ground state. These are different claims: a low-energy sample is not automatically an optimum or even a legal grouping.

Each full string contains **8 assignment bits followed by 6 slack bits**. Read left to right using the displayed variable order; do not reverse the string. Energies include the constant offset. We rank by the target QUBO, not by an assumed exact physical mapping, so this does not claim a ground state of the approximate atom Hamiltonian.

| Positions from the left (starting at 1) | Python indices | Meaning | Weights from left to right |
|---|---|---|---|
| 1-8 | 0-7 | Customer assignment bits | Not slack |
| 9-11 | 8-10 | Vehicle 0 slack: y_0_0, y_0_1, y_0_2 | 1, 2, 4 |
| 12-14 | 11-13 | Vehicle 1 slack: y_1_0, y_1_1, y_1_2 | 1, 2, 4 |

For example, `10010110110000` is shown as `10010110 | 110 | 000`: assignment bits, vehicle 0 slack, then vehicle 1 slack. `110` means **1 + 2 = 3 unused units**, not the ordinary left-to-right binary value 6. `000` means 0 unused units. The leftmost slack bit has weight 1 because the encoding uses little-endian slack bits.

Both exact and sampled tables show each vehicle's slack bits and decoded value. For a sampled string, this value only describes what its bits say: incorrect slack bits can disagree with the actual unused capacity, even when the customer grouping itself is feasible.

The full measured-bitstring table and ground-state evidence are saved under `outputs/first_demand/groundstates/`.

In [14]:
for instance in instances:
    target = build_qubo(instance)
    reference = json.loads((output / "references" / f"{instance.id}.json").read_text())
    exact = validate_small_qubo(instance, reference["clustering"]["cost"])
    run_path = output / "runs" / f"{instance.id}-quantum-global-{seed}.json"
    run = json.loads(run_path.read_text())
    if run["instance_hash"] != fingerprint(asdict(instance)):
        raise ValueError("Saved quantum shots do not match the current instance")
    if run["variable_order"] != target.variable_order:
        raise ValueError("Saved quantum bit order does not match the target QUBO")
    if run["status"] != "completed":
        print(instance.id, "has no completed quantum samples:", run["status"], run.get("error"))
        continue
    counts = run["counts"]
    if not counts or any(type(count) is not int or count <= 0 for count in counts.values()):
        raise ValueError("A completed quantum run must contain positive raw shot counts")
    total = sum(counts.values())
    if total != run["shot_budget"]:
        raise ValueError("Saved quantum shot counts do not match the shot budget")
    ground_strings = set(exact["ground_state_bitstrings"])
    measured = []
    for bitstring, count in counts.items():
        labels = decode(instance, bitstring)
        measured.append({
            "bitstring": bitstring,
            **bitstring_breakdown(instance, bitstring),
            "count": count, "probability": count / total,
            "target_energy": target.energy(list(map(int, bitstring))),
            "feasible": labels is not None, "customer_groups": labels,
            "exact_target_ground_state": bitstring in ground_strings,
        })
    measured.sort(key=lambda row: (row["target_energy"], -row["count"], row["bitstring"]))
    observed_ground = [row for row in measured if row["exact_target_ground_state"]]
    lowest_energy = measured[0]["target_energy"]
    lowest_samples = [row for row in measured
                      if np.isclose(row["target_energy"], lowest_energy, atol=1e-8, rtol=0)]
    print("Instance:", instance.id, "| bit positions, left to right:", target.variable_order)
    print("Exact target ground-state bitstrings:", exact["ground_state_bitstrings"])
    if observed_ground:
        print("Exact target ground states actually sampled:")
        display(pd.DataFrame(observed_ground))
    else:
        print("No exact target-QUBO ground state was sampled in this run.")
    print("Lowest-target-energy bitstrings actually sampled:")
    display(pd.DataFrame(lowest_samples))
    evidence_path = output / "groundstates" / f"{instance.id}-quantum-global-{seed}.json"
    save(evidence_path, {
        "instance_id": instance.id, "instance_hash": run["instance_hash"],
        "source_run_hash": fingerprint(run), "variable_order": target.variable_order,
        "bit_layout": bit_layout(instance, target),
        "exact_target_ground_energy": exact["minimum_energy"],
        "exact_target_ground_bitstrings": exact["ground_state_bitstrings"],
        "observed_ground_states": observed_ground,
        "lowest_sampled_energy": lowest_energy, "lowest_sampled_states": lowest_samples,
        "sampled_states": measured,
    })
    print("Full measured-bitstring table saved to:", evidence_path)

Instance: s-0 | bit positions, left to right: ['x_0_0', 'x_0_1', 'x_1_0', 'x_1_1', 'x_2_0', 'x_2_1', 'x_3_0', 'x_3_1', 'y_0_0', 'y_0_1', 'y_0_2', 'y_1_0', 'y_1_1', 'y_1_2']
Exact target ground-state bitstrings: ['10010110110000', '01101001000110']
No exact target-QUBO ground state was sampled in this run.
Lowest-target-energy bitstrings actually sampled:


,bitstring,assignment_bits,slack_bits,slack_bits_vehicle_0,slack_value_vehicle_0,slack_bits_vehicle_1,slack_value_vehicle_1,grouped_bitstring,count,probability,target_energy,feasible,customer_groups,exact_target_ground_state
0,00100101101100,00100101,101100,101,5,100,1,00100101 | 101 | 100,1,0.003906,5.54448,False,None,False


Full measured-bitstring table saved to: /workspaces/duquantum_hack_2026/track_a/outputs/first_demand/groundstates/s-0-quantum-global-0.json


## Interpretation and limits

| Formulation | Assignment atoms | Slack atoms | Extra auxiliaries | Total S atoms |
|---|---:|---:|---:|---:|
| Assignment-only estimate, not executed | 8 | 0 | 0 | 8 |
| Textbook / implemented binary slack | 8 | 6 | 0 | 14 |

No nonquadratic terms or negative pair coefficients are silently discarded.
Formulation correctness, embedding error, and finite-duration sampling quality
are different questions. No quantum speedup, exact embedding, DMM, M/L, noise
robustness, or time-step convergence is claimed. Continue to the results
notebook for shot-weighted gaps and the report.